# バージョン情報
## V3.0
   
   1                      id  : 顧客ID : int64   
   2                     Age  : 顧客の年齢 : str (データがぐちゃぐちゃなので整理)   
   3           TypeofContact  : 顧客への連絡・接触方法 : str   
   4                CityTier  : 都市層(1>2>3) : int64   
   5         DurationOfPitch  : 営業担当者による顧客への商品のセールス時間 : str   
   6              Occupation  : 顧客の職業 : str   
   7                  Gender  : 顧客の性別 : str   
   8  NumberOfPersonVisiting  : 予定している旅行の同行者の数	 : int64   
   9       NumberOfFollowups  : セールス後に営業担当者が行ったフォローアップの回数 : float64   
   10          ProductPitched : 営業担当者のセールスした商品の種類 : str   
   11   PreferredPropertyStar : 顧客の希望するホテルのランク : float64   
   12           NumberOfTrips : 顧客の年間旅行数 : str   
   13                Passport : パスポートの所持(0: 不所持、1: 所持) : int64   
   14  PitchSatisfactionScore : 営業担当者のセールストークに対する顧客の満足度 : int64   
   15             Designation : 顧客の役職 : str   
   16           MonthlyIncome : 顧客の月収 : str   
   17           customer_info : 顧客の情報のメモ(婚姻状況や車の有無、旅行への子どもの同伴の有無) : str 

====================================================================================
追加

   19             Age_20s_30s : 20代・30代限定フラグ　（1: 20〜39歳、0:それ以外）: int64


====================================================================================
消去
   
　18               Age_Group : 顧客の年齢を10s 20s.....50s 60s+　までグループ化： str 
   


# ライブラリインポート

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from kanjize import number2kanji, kanji2number, Number, KanjizeConfiguration, KanjizeZero, KanjizeStyle

# データ加工

In [24]:
def df_Data_Cleansing(df):
    
    #df["Age"]の値を整頓
    df["Age"] = df['Age'].str.strip("歳, 代, 際, 才")
    def convert(x):
        if pd.isna(x):
            return x 
        try:
            return int(x)                                 
        except ValueError:
            return kanji2number(str(x))                   
    df["Age"] = df["Age"].apply(convert)

    #欠損値を中央値に変換
    df["Age"] = df["Age"].fillna(df['Age'].median())

    #19 df["Age"]20歳以上、40歳未満
    df['Age_20s_30s'] = df['Age'].apply(lambda x: 1 if 20 <= x < 40 else 0)
    
    """
    #18 10s 20s.....50s 60s+　までグループ化
    bins = [0, 20, 30, 40, 50, 60, 100]
    labels = ["10s", "20s", "30s", "40s", "50s", "60s+"]
    df["Age_Group"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False) 
    
    
    
    
    """ 


    return df

In [25]:
df = df_Data_Cleansing(pd.read_csv('data/train.csv',encoding="utf-8")) 
df_test = df_Data_Cleansing(pd.read_csv('data/test.csv',encoding="utf-8"))

# 特徴量追加分

# 学習データ

In [28]:
df

,id,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,NumberOfTrips,Passport,PitchSatisfactionScore,Designation,MonthlyIncome,customer_info,ProdTaken,Age_20s_30s
0,0,50.0,Self Enquiry,2,900秒,Large Business,male,1.0,4.0,Basic,3.0,5,1,4,Executive,253905.0,未婚 車未所持 子供なし,1,0
1,1,56.0,Company Invited,1,14分,Salaried,Male,1.0,4.0,Standard,3.0,2,1,4,Senior Manager,404475.0,"離婚済み,車あり,子供無し",0,0
2,2,38.0,Self Enquiry,1,10分,Large Business,Female,1.0,3.0,Basic,3.0,4,0,4,Executive,278145.0,"結婚済み、自動車未所有,子供なし",1,1
3,3,37.0,Self Enquiry,2,1080秒,Small Business,female,1.0,3.0,Standard,4.0,1,0,5,Senior Manager,326805.0,離婚済み、車所持、子供無し,0,1
4,4,48.0,Company Invited,3,1020秒,Small Business,female,1.0,3.0,Basic,4.0,4,0,4,Executive,258435.0,独身／車所持／無子,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3484,3484,40.0,Self Enquiry,2,1560秒,Salaried,Male,2.0,3.0,Basic,3.0,3,0,1,Executive,258900.0,離婚済み 乗用車所持 子供なし,1,0
3485,3485,40.0,Self Enquiry,1,9分,Large Business,Male,3.0,3.0,Basic,5.0,5,0,3,Executive,260415.0,結婚済み、車所持、子供2人,0,0
3486,3486,31.0,Self Enquiry,1,840秒,Small Business,FEMALE,3.0,2.0,Standard,3.0,5,0,4,Senior Manager,317340.0,独身／車所持／こども1人,0,1
3487,3487,56.0,Company Invited,2,900秒,Salaried,Male,3.0,6.0,King,3.0,7,1,4,VP,527910.0,結婚済み、自動車未所有、子供2人,1,0


# テストデータ

In [29]:
df_test

,id,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,NumberOfTrips,Passport,PitchSatisfactionScore,Designation,MonthlyIncome,customer_info,Age_20s_30s
0,3489,48.0,Self Enquiry,2,13分,Small Business,Male,1.0,4.0,Super De|uxe,3.0,7,0,3,AVP,496950.0,結婚済み 車所持 子供なし,0
1,3490,30.0,Self Enquiry,2,12分,Small Business,Ｆｅｍａｌｅ,1.0,4.0,Standard,3.0,4,1,3,Senior Manager,月収30.0万円,結婚済み、車未所持、子供なし,1
2,3491,25.0,Self Enquiry,1,540秒,Salaried,Female,1.0,4.0,Basic,3.0,1,0,3,Executive,月収26.0万円,離婚済み、自動車未所有、子供なし,1
3,3492,21.0,Company Invited,2,420秒,Salaried,Male,1.0,4.0,Basic,4.0,1,0,3,Senior Manager,259875.0,離婚済み、自動車所有、子供なし,1
4,3493,41.0,Company Invited,1,7分,Salaried,MALE,1.0,4.0,Basic,3.0,1,0,4,Executive,268830.0,独身/車所持／子供なし,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3484,6973,41.0,Company Invited,1,14分,Small Business,Female,1.0,3.0,Basic,3.0,2,0,4,Executive,261840.0,結婚済み/車なし／子供無し,0
3485,6974,44.0,Company Invited,1,35分,Salaried,Male,3.0,5.0,Deluxe,3.0,3,0,3,Manager,349770.0,結婚済み、自家用車あり、子供1人,0
3486,6975,24.0,Self Enquiry,2,21分,Small Business,male,2.0,3.0,basic,3.0,2,0,3,Executive,月収27.0万円,独身、車未所持、子供なし,1
3487,6976,25.0,Self Enquiry,1,9分,Small Business,MALE,2.0,3.0,Basic,3.0,2,0,3,Executive,272430.0,結婚済み、車未所持、こども1人,1


# 学習

In [30]:
y = df["ProdTaken"]
x = df.drop(columns=["ProdTaken", "id"])

cat_cols = x.select_dtypes(include=['object','str']).columns #データの中から「文字列（objectやstr）」の列をすべて探し出す
x[cat_cols] = x[cat_cols].astype('category') # 2行目：探し出した列を、まとめて一括で「category型」に変換する

x

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,NumberOfTrips,Passport,PitchSatisfactionScore,Designation,MonthlyIncome,customer_info,Age_20s_30s
0,50.0,Self Enquiry,2,900秒,Large Business,male,1.0,4.0,Basic,3.0,5,1,4,Executive,253905.0,未婚 車未所持 子供なし,0
1,56.0,Company Invited,1,14分,Salaried,Male,1.0,4.0,Standard,3.0,2,1,4,Senior Manager,404475.0,"離婚済み,車あり,子供無し",0
2,38.0,Self Enquiry,1,10分,Large Business,Female,1.0,3.0,Basic,3.0,4,0,4,Executive,278145.0,"結婚済み、自動車未所有,子供なし",1
3,37.0,Self Enquiry,2,1080秒,Small Business,female,1.0,3.0,Standard,4.0,1,0,5,Senior Manager,326805.0,離婚済み、車所持、子供無し,1
4,48.0,Company Invited,3,1020秒,Small Business,female,1.0,3.0,Basic,4.0,4,0,4,Executive,258435.0,独身／車所持／無子,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3484,40.0,Self Enquiry,2,1560秒,Salaried,Male,2.0,3.0,Basic,3.0,3,0,1,Executive,258900.0,離婚済み 乗用車所持 子供なし,0
3485,40.0,Self Enquiry,1,9分,Large Business,Male,3.0,3.0,Basic,5.0,5,0,3,Executive,260415.0,結婚済み、車所持、子供2人,0
3486,31.0,Self Enquiry,1,840秒,Small Business,FEMALE,3.0,2.0,Standard,3.0,5,0,4,Senior Manager,317340.0,独身／車所持／こども1人,1
3487,56.0,Company Invited,2,900秒,Salaried,Male,3.0,6.0,King,3.0,7,1,4,VP,527910.0,結婚済み、自動車未所有、子供2人,0


In [31]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42)

train_data = lgb.Dataset(x_train,label=y_train)
valid_data = lgb.Dataset(x_test, label=y_test, reference=train_data)#

params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'metric': 'binary_logloss',
    'num_leaves': 31,
    'learning_rate': 0.05    
}

model = lgb.train(
    params,
    train_data,
#   valid_sets=[train_data, valid_data], #
    num_boost_round=100,
#   callbacks=[lgb.early_stopping(stopping_rounds=50)] 
)

[LightGBM] [Info] Number of positive: 406, number of negative: 2385
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000497 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 511
[LightGBM] [Info] Number of data points in the train set: 2791, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.145468 -> initscore=-1.770601
[LightGBM] [Info] Start training from score -1.770601


In [32]:
y_pred_proba = model.predict(x_test)
y_pred_proba[:10]

array([0.15870563, 0.00819344, 0.01782407, 0.68035809, 0.05546184,
       0.02460089, 0.01704072, 0.62552603, 0.00683695, 0.01359022])

In [33]:
auc = roc_auc_score(y_test, y_pred_proba)
print("roc-auc : ",auc)

roc-auc :  0.8309285442728607


# テスト

In [34]:
submit_x = df_test.drop(columns=["id"])
submit_x[cat_cols] = submit_x[cat_cols].astype('category')

In [35]:
submit_x

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,NumberOfTrips,Passport,PitchSatisfactionScore,Designation,MonthlyIncome,customer_info,Age_20s_30s
0,48.0,Self Enquiry,2,13分,Small Business,Male,1.0,4.0,Super De|uxe,3.0,7,0,3,AVP,496950.0,結婚済み 車所持 子供なし,0
1,30.0,Self Enquiry,2,12分,Small Business,Ｆｅｍａｌｅ,1.0,4.0,Standard,3.0,4,1,3,Senior Manager,月収30.0万円,結婚済み、車未所持、子供なし,1
2,25.0,Self Enquiry,1,540秒,Salaried,Female,1.0,4.0,Basic,3.0,1,0,3,Executive,月収26.0万円,離婚済み、自動車未所有、子供なし,1
3,21.0,Company Invited,2,420秒,Salaried,Male,1.0,4.0,Basic,4.0,1,0,3,Senior Manager,259875.0,離婚済み、自動車所有、子供なし,1
4,41.0,Company Invited,1,7分,Salaried,MALE,1.0,4.0,Basic,3.0,1,0,4,Executive,268830.0,独身/車所持／子供なし,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3484,41.0,Company Invited,1,14分,Small Business,Female,1.0,3.0,Basic,3.0,2,0,4,Executive,261840.0,結婚済み/車なし／子供無し,0
3485,44.0,Company Invited,1,35分,Salaried,Male,3.0,5.0,Deluxe,3.0,3,0,3,Manager,349770.0,結婚済み、自家用車あり、子供1人,0
3486,24.0,Self Enquiry,2,21分,Small Business,male,2.0,3.0,basic,3.0,2,0,3,Executive,月収27.0万円,独身、車未所持、子供なし,1
3487,25.0,Self Enquiry,1,9分,Small Business,MALE,2.0,3.0,Basic,3.0,2,0,3,Executive,272430.0,結婚済み、車未所持、こども1人,1


In [36]:
test_pred = model.predict(submit_x)
test_pred

array([0.05725566, 0.151037  , 0.35828964, ..., 0.48397304, 0.36394313,
       0.00713347], shape=(3489,))

# 提出ファイル、submit_x（テスト）生成

In [37]:
submission = pd.DataFrame({
    'id': df_test['id'],
    'y': test_pred
})

submission.to_csv('submission.csv', index=False, header=False)
submit_x.to_csv("submit_x_x_x_x_x_x.csv", index=False, header=False)

submission

,id,y
0,3489,0.057256
1,3490,0.151037
2,3491,0.358290
3,3492,0.532658
4,3493,0.220345
...,...,...
3484,6973,0.089094
3485,6974,0.015705
3486,6975,0.483973
3487,6976,0.363943
